# 5. The 43-motif CNN-CR, trained and tested start to end on one cell line (Fig. 2e)

How much expression signal do the 43 core motifs carry? The paper's test is a CNN-CR whose first-layer filters are **the 43 PWMs**: a promoter-only model (the ten promoter windows, one branch each), where the 43 filters are initialised with the PWMs and then trained with the rest of the network ("motif-initialised"; `code/posthoc_analysis/core_cnn_lib.py` holds the model, training loop and hyperparameters of the paper's script verbatim). This notebook runs it on **K562** from the shipped window FASTA: one-hot tensor -> gene split -> training -> held-out Spearman correlation, compared with the published K562 value (0.789) and with the paper's Fig. 2e medians (0.710 with the 43 core motifs versus 0.733 with the biosample's full vocabulary, over 116 biosamples).

Needs a GPU (about 3 min on an A6000; on a CPU about an hour) and nothing from Zenodo for K562. Any other biosample: its input pair from `fig1_training_inputs_all116`.

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/rewrite_paths.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


In [ ]:
# ---- CONFIGURATION ---------------------------------------------------------------------------------------------
BSSID = "BSS00762"
MOTIFS = f"{ENHANCER_ROOT}/notebook_runs/fig2/universal_regulator_motifs.meme"     # the 43 motifs from notebook 04 ...
if not os.path.exists(MOTIFS): MOTIFS = f"{REPO}/data/motifs/core_regulator_motifs_43.meme"   # ... or the released copy
INPUTS = os.path.join(REPO, "data", "k562_demo") if BSSID == "BSS00762" else f"{ENHANCER_ROOT}/fig1_training_inputs"
WORK = f"{ENHANCER_ROOT}/notebook_runs/fig2/{BSSID}"; os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, f"{CODE}/cnncr_model"); sys.path.insert(0, f"{CODE}/posthoc_analysis")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, logomaker
from cnncr_io import read_windows_fasta, read_expression
import core_cnn_lib as L
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device, "| motifs:", MOTIFS)

## 5.1 The 43 filters and the input
`parse_meme` reads the PWMs; the first convolution is one `Conv1d(4, 43, 8)` whose weights start as `PWM - 0.25` (a PWM-shaped filter with zero mean). The input is the promoter part of the tensor of notebook 01: slots 0-9 (TSS window and up to nine peak windows).

In [ ]:
motifs, names = L.parse_meme(MOTIFS); print(len(motifs), "motifs, widths", sorted(set(m.shape[0] for m in motifs)))
X, _ = read_windows_fasta(f"{INPUTS}/{BSSID}.windows.fa.gz", n_genes=L.GENES); genes, y = read_expression(f"{INPUTS}/{BSSID}.expression.tsv")
X = X[:, :L.NUM_BRANCHES]; print("promoter tensor:", X.shape)
fig, axes = plt.subplots(1, 6, figsize=(12, 1.3))
for ax, i in zip(axes, range(6)): logomaker.Logo(logomaker.transform_matrix(pd.DataFrame(motifs[i].astype(float), columns=list("ACGT")), from_type="probability", to_type="information"), ax=ax); ax.set_title(names[i], fontsize=8); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 5.2 Gene split and training
The protocol of `05_cnn_comparison.py`: genes permuted with seed 42, 10 % held out (`random_state = 42`), fold 0 of a 10-fold split of the rest for early stopping; Adam (lr 10⁻⁴), MSE + L2 on the dense layers, batch 64, at most 100 epochs, patience 10.

In [ ]:
np.random.seed(L.SEED); torch.manual_seed(L.SEED)
perm = np.random.permutation(L.GENES); Xp, yp = X[perm], y[perm].astype(np.float32)
Xtr_full, Xte, ytr_full, yte = L.train_test_split(Xp, yp, test_size=0.1, random_state=L.SEED)
tr_idx, val_idx = next(iter(L.KFold(n_splits=10, shuffle=True, random_state=L.SEED).split(Xtr_full)))
Xtr, Xval, ytr, yval = Xtr_full[tr_idx], Xtr_full[val_idx], ytr_full[tr_idx], ytr_full[val_idx]
print("train", len(ytr), "| validation", len(yval), "| held-out test", len(yte))
T = lambda a: torch.tensor(a).float()
tr_loader = L.DataLoader(L.TensorDataset(T(Xtr), T(ytr).unsqueeze(1)), batch_size=L.BATCH, shuffle=True)
val_loader = L.DataLoader(L.TensorDataset(T(Xval), T(yval).unsqueeze(1)), batch_size=len(yval), shuffle=False)
model = L.MultiBranchModel(motifs, len(motifs), "universal", n_branches=L.NUM_BRANCHES).to(device)
w0 = model.branches[0].motif_conv.conv.weight.detach().cpu().clone() if hasattr(model.branches[0], "motif_conv") else None
t0 = time.time(); model = L.train_model(model, tr_loader, val_loader, device); print(f"training: {(time.time()-t0)/60:.1f} min")
torch.save(model.state_dict(), f"{WORK}/{BSSID}_universal_model.pth")

## 5.3 Held-out performance

In [ ]:
res = {k: L.corr_S(v, L.predict_all(model, T(Xs), device)) for k, v, Xs in (("train", ytr, Xtr), ("validation", yval, Xval), ("test", yte, Xte))}
for k, v in res.items(): print(f"{k:11s} Spearman {v:.3f}")
np.save(f"{WORK}/{BSSID}_universal_test_S.npy", np.array([res["test"]]))
print(f"\npublished K562, 43 core motifs: {float(np.load(f'{REPO}/data/k562_sample/BSS00762_universal_test_S.npy')):.3f} | K562 full vocabulary (notebook 01, ten models): 0.806 | paper Fig. 2e medians over 116 biosamples: 0.710 (core) vs 0.733 (full)")
pred = L.predict_all(model, T(Xte), device); yt, ypd = yte / np.log2(10), pred / np.log2(10)
fig, ax = plt.subplots(figsize=(3, 3)); ax.scatter(yt, ypd, s=3, color="#7E86B8", alpha=0.5, rasterized=True); ax.plot([0, 3], [0, 3], ls="--", color="#66737c", lw=0.8)
ax.set_xlabel("measured (log10 FPKM)"); ax.set_ylabel("predicted (log10 FPKM)"); ax.set_title(f"K562, 43-motif CNN-CR, ρ = {res['test']:.3f}", fontsize=8); ax.set_box_aspect(1)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); fig.savefig(f"{WORK}/{BSSID}_core43_scatter.png", dpi=200, transparent=True); plt.show()

## 5.4 Did the filters stay the core motifs?
The filters are initialised with the PWMs and trained; the change is small: the trained filters remain recognisable versions of the motifs they started from.

In [ ]:
br = model.branches[0]; conv = [m for m in br.modules() if isinstance(m, torch.nn.Conv1d)][0]
w1 = conv.weight.detach().cpu(); w0 = torch.stack([torch.tensor(m - 0.25).T for m in motifs])
d = (w1 - w0).abs().amax(dim=(1, 2)); print(f"max |change| per filter: median {d.median():.3f}, max {d.max():.3f} (PWM entries are in 0-1)")
i = int(d.argmax()); fig, axes = plt.subplots(1, 2, figsize=(4.5, 1.3))
for ax, w, t in zip(axes, (w0[i], w1[i]), ("initial", "trained")):
    logomaker.Logo(pd.DataFrame(w.T.numpy().astype(float), columns=list("ACGT")), ax=ax); ax.set_title(f"{names[i]} ({t}): filter weights", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()